In [1]:
%run ../stage1.py \
    --sc_file "/mnt/d/ST_Graduation_Project_data/database/GSE225857/GSE225857_sub_SC.h5ad" \
    --st_file "/mnt/d/ST_Graduation_Project_data/database/GSE225857/GSE225857_C2_ST.h5ad" \
    --n_epochs 300 \
    --resolution 2 \
    --loss_type zinb \
    --beta 0.1 \
    --lambda_mmd 1.0 \
    --top_n_per_type 100 \
    --latent_dim 256 \
    --output_dir ../deconv_results/GSE225857_C2 \
    --precomputed_marker_file "../deconv_results/GSE225857_L1/final_genes.txt"

/home/mwc/miniconda3/envs/dl/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Using device: cpu
Stage 1 Training: VAE (SC + ST, Marker Genes)
Configuration:
   Marker genes per type: 100
   Leiden resolution: 2.0
   Precomputed marker genes: ../deconv_results/GSE225857_L1/final_genes.txt
   Batch size: 256
   Epochs: 300
   Learning rate: 0.0005
   Beta (KL weight): 0.1
   Hidden dims: [512, 256]
   Latent dim: 256
   Loss type: ZINB
   Lambda MMD: 1.0
   Dual Decoder: True
   Aggregation method: mean
Loading datasets...
   Loading SC: /mnt/d/ST_Graduation_Project_data/database/GSE225857/GSE225857_sub_SC.h5ad
   SC shape: (67119, 16576)
   SC avg counts/cell: 4331.0 (from X)
   Loading ST: /mnt/d/ST_Graduation_Project_data/database/GSE225857/GSE225857_C2_ST.h5ad
   ST shape: (3305, 17943)
   Common genes: 15064
   SC final: (67119, 15064)
   ST final: (3305, 15064)
Using precomputed marker genes from: ../deconv_results/GSE225857_L1/final_genes.txt
Loading marker genes from file: ../deconv_results/GSE225857_L1/final_genes.txt
   Loaded 1703 marker genes
Saved clu

VAE Training:  70%|███████   | 210/300 [51:18<21:59, 14.66s/epoch, Train=1474.6573, Recon=1469.8280, KL=48.2908, MMD=0.0397, Test=1492.7248] 



⚠️ Early stopping triggered at epoch 211/300
   Best test loss: 1492.5818, Current test loss: 1492.7248
   No improvement for 20 consecutive epochs
📊 Computing cluster centers using ALL SC data (train + test)...
   ALL SC data: (67119, 1703)
   Number of clusters: 52
   Computed embeddings shape: (67119, 256)
Computing cluster centers with mean aggregation...
      Cluster 0: 4022 cells (mean aggregation)
      Cluster 1: 3789 cells (mean aggregation)
      Cluster 2: 2125 cells (mean aggregation)
      Cluster 3: 2000 cells (mean aggregation)
      Cluster 4: 1731 cells (mean aggregation)
      Cluster 5: 1676 cells (mean aggregation)
      Cluster 6: 1658 cells (mean aggregation)
      Cluster 7: 1648 cells (mean aggregation)
      Cluster 8: 1583 cells (mean aggregation)
      Cluster 9: 1564 cells (mean aggregation)
      Cluster 10: 1535 cells (mean aggregation)
      Cluster 11: 1359 cells (mean aggregation)
      Cluster 12: 2996 cells (mean aggregation)
      Cluster 13: 1345 

In [2]:
%run ../stage2.py \
    --st_file "/mnt/d/ST_Graduation_Project_data/database/GSE225857/GSE225857_C2_ST.h5ad" \
    --stage1_model_path "../deconv_results/GSE225857_C2/final_vae.pth" \
    --output_dir "../deconv_results/GSE225857_C2/" \
    --gat_hidden_dim 512 \
    --gat_layers 4 \
    --lr 1e-4 \
    --loss_lambda_mse 0.05 \
    --loss_lambda_cosine 2 \
    --loss_lambda_pearson 2 \
    --loss_lambda_reg 0.1 \
    --loss_lambda_sparse 0.01 \
    --loss_lambda_proportion 0.1 \
    --k_spatial 10 \
    --k_celltype 20 \
    --batch_size 512 \
    --n_epochs 500 \
    --weight_threshold 0.05

Sample name: GSE225857_C2
Stage 1 model: ../deconv_results/GSE225857_C2/final_vae.pth
Output directory: ../deconv_results/GSE225857_C2/
Device: cpu
Weight threshold: 0.05
Loading pretrained VAE Encoder...
   VAE architecture: 1703 -> 256
   Output type: zinb
   Architecture: Dual Decoder (SC/ST-specific)
   ✓ Loaded 67119 cell cluster labels from checkpoint
Loading cluster data from: ../deconv_results/GSE225857_C2/final_vae_cluster_data.npz
   Cluster prototypes: torch.Size([52, 256])
   Cluster expressions (marker): torch.Size([52, 1703])
   Cluster expressions (all genes): 52 × 15064
   Loaded celltype mapping: 52 clusters
   Average cell counts: 4331.0
Loaded all genes list: 15064 genes
VAE Encoder loaded: 1703 -> 256
Cell type clusters: ['0', '1', '10', '11', '12', '13', '14', '15', '16', '17', '18', '19', '2', '20', '21', '22', '23', '24', '25', '26', '27', '28', '29', '3', '30', '31', '32', '33', '34', '35', '36', '37', '38', '39', '4', '40', '41', '42', '43', '44', '45', '46', '

GAT Training:  27%|██▋       | 134/500 [16:05<43:56,  7.20s/epoch, Total=25.0098, Pearson=0.4900, MSE=465.4775, Cosine=0.3130, P_pat=10, M_pat=13, C_pat=10]



⚠️ Early stopping triggered at epoch 135/500
   All three core losses stopped improving:
      Pearson: best=0.4873, current=0.4900, no improvement for 10 epochs
      MSE: best=460.8909, current=465.4775, no improvement for 13 epochs
      Cosine: best=0.3102, current=0.3130, no improvement for 10 epochs
Evaluating model results...
Applying weight threshold: 0.05
   Non-zero elements: 66100 -> 19278 (11.2%)
Saving deconvolution results...
Generating deconvolution expression matrices...
   Reconstructing all gene expression...
   Saved: ../deconv_results/GSE225857_C2//GSE225857_C2_reconstructed_all_genes.csv
   Cell type composition...
   Found duplicate celltype names: ['cDC', 'Tumor', 'Monocyte', 'Macrophage', 'Fibroblast', 'T cell', 'NK cell', 'B cell / Plasma', 'Endothelial']. Merging corresponding cluster columns by summing weights.
   Columns before: 52, after merge: 11
   Saved cell composition (celltype): ../deconv_results/GSE225857_C2//GSE225857_C2_cell_composition.csv
   Sav